# Step 7: 评测方法论（downstream lm_eval + 性能/显存）— 验证调优收益

**目标**：建立完整的量化模型评测方法论，并**验证 s5/s6 调优闭环的收益**。对比维度从「三方法（FP8/AWQ/SmoothQuant）」重构为**调优前→后 + 上界四向对比**：`s5_baseline`（全量化起点，掉点最狠）→ `s5_tuned`（s5 ignore 调优后）→ `s6_final`（s6 最优 α+group_size，调优终点）→ **FP16 上界**。三维度合成四维对比表（质量 PPL+downstream | 显存 | 吞吐 | TTFT），是整个 M3 工业调优闭环的**裁决/finale**——证明调优结果真有效。

**对应 OUTLINE 课时**：3.7 评测方法论（~55 分钟）。

> **双 env 提醒**：本 notebook 在 `steps/vllm` 子项目跑（vLLM + lm_eval env）。PPL 维的数据由 s1-s6 在 `steps/quant` env 产出、存共享 `out/`，本节**只读不重算**。对比的四向产物都在 `out/`（s5_baseline/s5_tuned/s6_final）+ `models/Qwen2.5-7B-Instruct`（FP16 上界）。


## 学完应能讲清（学完本节应能口头回答）

1. s7 为什么**验证调优收益**而不是对比三方法？四向对比（baseline→tuned→final→FP16）分别代表什么？（baseline=全量化掉点起点；tuned=s5 ignore 后；final=s6 最优 α 后；FP16=精度上界——展示调优闭环把掉点一步步追回）
2. 量化模型评测为什么**至少要测 PPL + downstream 两类**？（PPL 是通用语言建模代理，downstream 是真实任务；OUTLINE 易错：PPL 几乎不变但 GSM8K 可能掉）
3. `lm_eval --model vllm` 的关键参数有哪些？为什么量化前后必须用**相同 seed/prompt/num_fewshot**？（否则对比不可信）
4. 测显存为什么推荐读 vLLM `/metrics` 的 `kv_cache_usage_perc`，而**差值法（nvidia-smi）会高估**？（nvidia-smi 显示进程已映射显存，含预分配非实际 KV）
5. `lm_eval[vllm]` extra 为什么不在 uv.lock、要手动 `uv pip install --python ./.venv/bin/python`？（与 vLLM wheel 自带 transformers 可能互锁）


In [ ]:
%%capture
import pathlib, os, json, subprocess, shlex
import ipytest
try:
    ipytest.autoconfig()
except Exception:
    pass  # nbconvert 非交互上下文（无 IPython shell）
# lm_eval 可能未装（坑：lm_eval[vllm] extra 不在 uv.lock，需手动 uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'）
try:
    import lm_eval
    HAS_LM_EVAL = True
except ImportError:
    HAS_LM_EVAL = False
    print("[warn] lm_eval 未安装。先在 steps/vllm 跑：uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'")

In [ ]:
# Setup cell（双 env：本 notebook 在 steps/vllm 子项目跑；模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
# s7 跨两 env：PPL 数据从 steps/quant 产物读（out/ 共享）、downstream 在本 vllm env 跑 lm_eval。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"   # FP16 上界
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
QUANT_OUT        = OUT_ROOT  # 同一个 out/（模块根共享）
# === 四向对比产物（调优前→后 + 上界）===
COMPARISON_DIRS = {
    "baseline": OUT_ROOT / "s5_baseline",   # 全量化起点（掉点最狠）
    "tuned":    OUT_ROOT / "s5_tuned",      # s5 ignore 调优后
    "final":    OUT_ROOT / "s6_final",      # s6 最优 α+group_size（调优终点）
    "FP16":     MODEL_DIR,                  # 精度上界
}
print("MODULE_ROOT =", MODULE_ROOT)
print("本 env:", "vllm" if pathlib.Path('.venv/bin/vllm').exists() else "?", "| lm_eval:", HAS_LM_EVAL)
print("=== 四向对比产物存在性 ===")
for name, d in COMPARISON_DIRS.items():
    print(f"  {name:9s}: {'@ '+str(d) if d.exists() else '缺失 '+str(d)+'（先跑对应 quant env L3 生成）'}")


## 原理：评测三维度（OUTLINE 3.7）+ 调优收益验证

量化模型的「好」要**多维**测，单看任一维都会误判：

| 维度 | 工具 | 测什么 | env |
|---|---|---|---|
| **质量-PPL** | transformers forward | 语言建模困惑度（通用代理，快）| quant（s1-s6 已测）|
| **质量-downstream** | `lm_eval --model vllm` | gsm8k/mmlu/ceval 真实任务准确率 | vllm（本节）|
| **性能** | `vllm bench serve/throughput/latency` | 吞吐 + TTFT + 延迟 | vllm |
| **显存** | vLLM `/metrics`（Prometheus）| 权重 vs KV-Cache 占用 | vllm |

**对比维度重构**（spec §4 s7）：本节不再对比 FP8/AWQ/SmoothQuant 三方法（那是 M1/M2 的选型话题），而是**验证本模块调优闭环的收益**——同一 SmoothQuant 方案，调优前（baseline 全量化）vs 调优中（tuned，s5 ignore）vs 调优后（final，s6 最优 α）vs 上界（FP16）。四个点连成一条「调优把掉点追回」的曲线，证明 s5/s6 的工程价值。其它量化方案（FP8/AWQ）的对比见 M1/M2。

**关键易错点**（OUTLINE 3.7）：
- **PPL 几乎不变但 downstream 可能掉**：OUTLINE 3.8 明确——代码/数学任务对量化更敏感。所以**必须 downstream 实测**，PPL 不能当唯一判据。

  **为什么代码/数学任务对量化更敏感**（「任务精度更易掉」的因果，**不是**性能/吞吐维）：
  - **PPL 是单步平均困惑度**：它对整个词表求一次 cross-entropy 再对所有位置取平均。少量 token 的局部塌缩会被海量的正常 token **平滑掉**——整段文本的 PPL 几乎不动。
  - **代码/数学任务是多步符号推理链**：gsm8k 一道题要连续推导 5-20 步，**每一步的输出都是下一步的输入**。低 bit 权重误差在这样一条**长链**上会**累积放大**——第 3 步一个 token 算错，后面所有步骤都建立在错的前提上，整道题判错。
  - **结论**：PPL 把这类「局部塌缩」平滑成几乎不变的全局平均，而 gsm8k 的「最终答案 exact_match」是 **0/1 裁决**、不容平滑。故 PPL≈不变、gsm8k 掉几个点完全合理——这也是为什么调优收益要在 downstream 上验证，不只看 PPL。

- **显存拆分要用 `/metrics` 不是 nvidia-smi**：`nvidia-smi` 显示进程已映射显存（含预分配），会**高估** KV-Cache。推荐读 `vllm:kv_cache_usage_perc` / `num_gpu_blocks`。
- **性能要用 server 端 `/metrics` + `vllm bench serve`**：V1 引擎下用 `llm.generate()` 拿 per-request TTFT **不可靠**。
- **lm_eval 必须固定 seed/prompt/num_fewshot**：量化前后不一致则对比无意义。

**`lm_eval[vllm]` 安装坑**（OUTLINE 附录 B）：extra 不进 uv.lock（与 vLLM wheel 自带 transformers 可能互锁），必须 `uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'`。


### 端到端：评测在 M3 调优闭环的位置（finale）

s5（精度过线）→ s6（α 调参最优）→ **s7（裁决）**——证明调优结果真有效。s7 把 baseline→tuned→final→FP16 四个点放一起，验证「调优闭环把全量化的掉点一步步追回、逼近 FP16 上界」。三维度（PPL/downstream/性能/显存）合成四维对比表，是整个 M3 工业调优闭环的交付物（OUTLINE 3.8 四维对比表的输入）。


## 亲手摸一摸：lm_eval 任务清单 + metrics 字段名

看 lm_eval 支持哪些任务、结果 dict 长什么样——为构造评测命令打底。


In [ ]:
## 摸一摸：lm_eval 任务 & 结果结构（不真跑，只看 API）
if HAS_LM_EVAL:
    from lm_eval.tasks import TaskManager
    tm = TaskManager()
    all_tasks = tm.all_tasks
    wanted = ['gsm8k', 'mmlu', 'ceval-valid', 'cmmlu']
    present = [w for w in wanted if any(w in t for t in all_tasks)]
    print(f"lm_eval 任务总数 = {len(all_tasks)}")
    print(f"课程关心的任务命中: {present}")
    print("\n典型 gsm8k 结果字段: results['gsm8k']['exact_match,none'] (取值 0-1)")
    print("典型 mmlu 结果字段:  results['mmlu']['acc,none'] (取值 0-1)")
else:
    print("lm_eval 未装，跳过摸一摸（先 uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'）")


## 本步填空（2 个：downstream 命令构造 + PPL 维聚合）

1. **`run_downstream_eval(model_path, tasks, num_fewshot, limit)`**（判断型）—— 构造并（可选）执行 lm_eval 评测命令，返回结果。**为什么这么设计（填前先想）**：downstream 评测的参数选择（task、num_fewshot、limit）是判断决策——gsm8k 要 5-shot、limit 调优时小（250）最终大（全量）。本函数封装「选 task + 组参数 + 执行 + 抽指标」全流程，是 finale 的核心。本节用它对四个对比产物跑 downstream，看调优收益在真实任务上的体现。
2. **`read_ppl_from_artifacts(out_root, method_tags)`** —— 从 quant env 产物（共享 `out/`）读各对比点的 PPL，合成 PPL 维对比。**为什么这么设计**：s7 不重算 PPL（quant env 已算），只读共享 `out/` 聚合——这是双 env 架构的精髓（PPL 留 quant、downstream 留 vllm，产物通过 out/ 交接）。聚合对象从「三方法」改为「四对比点」（baseline/tuned/final/FP16）。


In [ ]:
def run_downstream_eval(model_path, tasks=('gsm8k',), num_fewshot=5, limit=250, execute=True):
    """判断型：构造 lm_eval 下游评测命令；execute=True 时真跑（需 GPU + lm_eval[vllm]），
    execute=False 只返回命令字符串（L1/L2 测命令构造逻辑）。
    返回 {'command': str, 'results': dict | None}。

    为什么这么设计（填前先想）：
    - task 选择是判断：gsm8k（数学）对量化最敏感、最能暴露掉点；mmlu/ceval 测通识。
    - num_fewshot：gsm8k 标配 5-shot（OUTLINE 3.5 命令）；limit=250 调优够、最终全量。
    - add_bos_token=True：Qwen2.5 评测惯例（OUTLINE 3.5 命令示例）。
    - 固定 seed（lm_eval 默认 random_seed=0）保证四向对比产物可比。
    命令模板（OUTLINE 3.5）：
      lm_eval --model vllm --model_args pretrained=<path>,add_bos_token=True \
        --tasks <tasks> --num_fewshot <n> --limit <limit>
    """
    # TODO: 1) tasks 是 tuple，join 成逗号串 'gsm8k,mmlu'。
    #       2) 构造命令：lm_eval --model vllm
    #          --model_args pretrained={model_path},add_bos_token=True
    #          --tasks {tasks_str} --num_fewshot {num_fewshot} --limit {limit}
    #          用 shlex.join 或空格拼接（注意 model_args 内逗号无空格）。
    #       3) 若 execute：subprocess.run(cmd, shell=True, capture_output=True, text=True) 跑；
    #          解析 stdout 找 'Results:' 后的 json，或调 lm_eval.simple_evaluate（更稳）。
    #          execute=False 时 results=None。
    #       4) 返回 {'command': cmd, 'results': results}。
    #   提示：L1/L2 测 execute=False（只验命令构造）；L3 execute=True。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（run_downstream_eval）——填完立即单独跑此 cell 验证（不依赖 read_ppl_from_artifacts）。

def test_run_downstream_eval_command_construction():
    out = run_downstream_eval("/tmp/model", tasks=('gsm8k', 'mmlu'), num_fewshot=5, limit=250, execute=False)
    cmd = out['command']
    assert 'lm_eval --model vllm' in cmd
    assert 'pretrained=/tmp/model,add_bos_token=True' in cmd, "model_args 应含 path + add_bos_token"
    assert '--tasks gsm8k,mmlu' in cmd, "tasks 应逗号 join"
    assert '--num_fewshot 5' in cmd and '--limit 250' in cmd
    assert out['results'] is None, "execute=False 时 results 应为 None"

def test_run_downstream_eval_single_task():
    out = run_downstream_eval("/tmp/m", tasks=('gsm8k',), execute=False)
    assert '--tasks gsm8k' in out['command']


In [ ]:
def read_ppl_from_artifacts(out_root, method_tags):
    """从 quant env 产物（共享 out/）读各对比点的 PPL，返回 {tag: ppl}。
    聚合对象（spec §4 s7 重构）：四向对比点——
      'baseline' -> s5_baseline（全量化起点）/ s5_pareto_curve.json 的 k=0 点。
                   取 curve 里 k 最小的点（通常 k=0），建议用 min(curve, key=lambda x: x[0]) 而非索引首位。
      'tuned'    -> s5_tuned（s5 ignore 调优后）/ 先读 s5_tuned_recipe.json 的 k_star（拐点），
                   再在 s5_pareto_curve.json 的 curve 里取 k==k_star 的 PPL（与 s5_tuned 目录同模型）。
      'final'    -> s6_alpha_scan.json 的最优 α 点（PPL 最低）
      'FP16'     -> FP16 上界（产物无 PPL 记录，记 None，由 L3 真跑补）
    每个 tag 的 PPL 读取顺序：先该 tag 专属 json，否则记 None（友好降级）。

    为什么这么设计（填前先想）：s7 不重算 PPL——PPL 是 quant env（transformers forward）
    的活，s1/s5/s6 已算好存 out/。s7 只聚合做对比维度，证明调优收益。这是双 env 架构的
    交接点：quant env 产 PPL → out/ → vllm env 读 → 合成 finale 表。
    tuned 用 k_star 而非 PPL 最低点：s5_tuned 目录存的是拐点 k* 模型，PPL 最低点可能对应
    最大 k（全回退），与 s5_tuned 不是同一模型——数据不自洽。
    """
    # TODO: out_root 是 pathlib.Path，method_tags 是 ['baseline','tuned','final','FP16']。
    #       对每个 tag：
    #       1) 'baseline': 读 out_root/'s5_pareto_curve.json' 的 curve，取 k 最小的点（通常 k=0 全量化）的 PPL；
    #          建议用 min(curve, key=lambda x: x[0]) 而非索引首位（不依赖 k 升序隐含约定）。
    #          文件不存在则 None。
    #       2) 'tuned': 先读 out_root/'s5_tuned'/'s5_tuned_recipe.json' 的 k_star；
    #          再读 out_root/'s5_pareto_curve.json' 的 curve，取 k==k_star 的 PPL。
    #          若 recipe 不存在则回退到取 curve 的 PPL 最低点（友好降级）。
    #          若 curve 文件不存在则 None。
    #       3) 'final': 读 out_root/'s6_alpha_scan.json' 的 alpha_scan，取 PPL 最小（最优 α）；
    #          文件不存在则 None。
    #       4) 'FP16': 无 json 产物，恒 None（FP16 上界 PPL 由 L3 真跑补）。
    #       5) 其它未知 tag：None。
    #       返回 {tag: ppl_or_None}。注意 json 异常/格式不符也降级为 None（友好）。
    raise NotImplementedError

In [ ]:
%%ipytest -qq
# L1 测试（read_ppl_from_artifacts）——填完立即单独跑此 cell 验证（不依赖 run_downstream_eval）。

def test_read_ppl_from_artifacts_missing_returns_none(tmp_path):
    # 无任何产物 -> 全 None（友好降级）
    out = read_ppl_from_artifacts(tmp_path, ['baseline', 'tuned', 'final', 'FP16'])
    assert out == {'baseline': None, 'tuned': None, 'final': None, 'FP16': None}

def test_read_ppl_from_artifacts_baseline_tuned_from_pareto(tmp_path):
    # baseline=k=0（全量化，PPL 最高）；tuned=拐点 k* 的 PPL（从 s5_tuned_recipe.json 读 k_star）
    import json, pathlib
    json.dump({'curve': [(0, 12.0), (2, 9.0), (4, 7.5)]},
              open(tmp_path / 's5_pareto_curve.json', 'w'))
    # 创建 s5_tuned_recipe.json 指定拐点 k_star=4（tuned PPL 应取 k=4 的 7.5）
    (tmp_path / 's5_tuned').mkdir()
    json.dump({'k_star': 4}, open(tmp_path / 's5_tuned' / 's5_tuned_recipe.json', 'w'))
    out = read_ppl_from_artifacts(tmp_path, ['baseline', 'tuned'])
    assert out['baseline'] == 12.0, "baseline 应取最小 k（k=0 全量化）的 PPL"
    assert out['tuned'] == 7.5, "tuned 应取拐点 k_star=4 的 PPL（与 s5_tuned 目录同模型）"

def test_read_ppl_from_artifacts_final_from_alpha_scan(tmp_path):
    import json
    json.dump({'alpha_scan': [{'alpha': 0.0, 'ppl': 10.0},
                              {'alpha': 0.8, 'ppl': 7.2},
                              {'alpha': 1.0, 'ppl': 9.5}],
               'best_alpha': 0.8}, open(tmp_path / 's6_alpha_scan.json', 'w'))
    out = read_ppl_from_artifacts(tmp_path, ['final'])
    assert out['final'] == 7.2, "final 应取 α 扫描的最优（PPL 最低）"

def test_read_ppl_from_artifacts_fp16_always_none(tmp_path):
    # FP16 上界无 json 产物，恒 None
    out = read_ppl_from_artifacts(tmp_path, ['FP16'])
    assert out['FP16'] is None

## L2（CPU）：构造评测命令 + 读 PPL 维聚合（不真跑 lm_eval）

L1/L2 验命令构造 + 产物读取逻辑（不需 GPU/lm_eval[vllm]）。造合成 quant 产物（s5_pareto_curve + s6_alpha_scan），验证 `read_ppl_from_artifacts` 正确聚合**四向对比**的 PPL 维——展示 baseline（掉点最狠）→ tuned → final 的精度追回曲线。


In [ ]:
## L2：合成 quant 产物，跑四向 PPL 维聚合
import tempfile, json
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    # 模拟 s5 产的 Pareto 曲线（非单调：k=4 拐点 PPL=8.0，k=6 反弹 8.2 —— 验证取拐点≠取最低点）
    json.dump({'curve': [(0, 11.8), (2, 9.5), (4, 8.0), (6, 8.2)]},
              open(td / 's5_pareto_curve.json', 'w'))
    # s5_tuned 的拐点配方：k_star=4（与 s5_tuned 目录存同一模型）
    (td / 's5_tuned').mkdir()
    json.dump({'k_star': 4, 'ignore': ['lm_head', 'layer_0', 'layer_1', 'layer_2', 'layer_3']},
              open(td / 's5_tuned' / 's5_tuned_recipe.json', 'w'))
    # 模拟 s6 产的 α 扫描（final 最优 α 进一步压低 PPL）
    json.dump({'alpha_scan': [{'alpha': 0.0, 'ppl': 7.8},
                              {'alpha': 0.8, 'ppl': 7.2},
                              {'alpha': 1.0, 'ppl': 7.9}],
               'best_alpha': 0.8}, open(td / 's6_alpha_scan.json', 'w'))
    tags = ['baseline', 'tuned', 'final', 'FP16']
    ppls = read_ppl_from_artifacts(td, tags)
    print("四向 PPL 维（从 quant 产物读，调优前→后→上界）：")
    for t in tags:
        print(f"  {t:9s} PPL={ppls[t]}")
    assert ppls['baseline'] == 11.8, "baseline=全量化起点 PPL 最高"
    assert ppls['tuned'] == 8.0, "tuned=拐点 k*=4 的 PPL（8.0），而非最低点（8.0 恰为最低但取的是 k_star 对应值）"
    assert ppls['final'] == 7.2, "final=s6 最优 α 后 PPL 进一步降"
    assert ppls['FP16'] is None, "FP16 上界无产物（L3 真跑补）"
    print("\n注意：curve 在 k=6 反弹到 8.2 > k=4 的 8.0——说明取「PPL 最低点」与「拐点 k*」可能不对应。")
    print("tuned 从 recipe 读 k_star=4，取 curve[2]=(4,8.0)，而非 curve 末点 (6,8.2)。")
    print("调优收益（PPL 维）：baseline 11.8 -> tuned 8.0 -> final 7.2（逼近 FP16 上界）")

# 命令构造（不执行）
print("\n各对比点 downstream 评测命令（gsm8k 5-shot）：")
for name, path in [('baseline','/out/s5_baseline'), ('tuned','/out/s5_tuned'),
                   ('final','/out/s6_final'), ('FP16','/models/qwen-fp16')]:
    out = run_downstream_eval(path, tasks=('gsm8k',), num_fewshot=5, limit=250, execute=False)
    print(f"  {name:9s}: {out['command']}")
print("\nL2 通过：命令构造 + 四向 PPL 维聚合逻辑正确（调优收益趋势 baseline>tuned>final）。")

## L3（H200，GPU + SKIP_L3 双守卫）：真 7B 四向调优收益验证 finale

在 7B 上对四个对比产物（baseline/tuned/final/FP16）跑 downstream（gsm8k 5-shot）+ 读 PPL 维 + 抓显存/吞吐，合成 OUTLINE 3.8 的四维对比表。这是 M3 工业调优闭环的 finale——**验证 s5/s6 调优在真实任务上的收益**（baseline 掉点 → tuned/final 追回 → 逼近 FP16）。

> 缺产物时友好降级（从对比表排除缺失的 tag，不崩）。FP16 上界 PPL 在本 L3 真跑补上。

> L3 双守卫：除 GPU 外，reviewer 执行验证设 `SKIP_L3=1` 跳过（lm_eval 真跑慢）；真人/学员跑不设，L3 实证。


In [ ]:
import torch, os, json
def run_l3_four_dim_eval():
    # 四向对比产物：调优前(baseline) -> 中(tuned) -> 后(final) -> 上界(FP16)
    # 设计 §5：本节读 quant env 已产的模型目录做 finale，不在 vllm env 重量化
    # （llmcompressor 仅装在 quant env；vllm env 无该依赖，见双 env 提醒）。
    available = {tag: p for tag, p in COMPARISON_DIRS.items() if p.exists()}
    missing = [t for t in COMPARISON_DIRS if t not in available]
    if missing:
        print(f"[warn] 以下对比点产物不存在，从四维表排除（先跑对应 quant env L3 生成）：{missing}")
        print("  baseline: quant env s5 L3 -> out/s5_baseline")
        print("  tuned:    quant env s5 L3 -> out/s5_tuned")
        print("  final:    quant env s6 L3 -> out/s6_final")
    if not available:
        print("[abort] 无任何对比点产物——至少跑一个 quant env L3 步骤再回 s7。")
        return
    # 维1: PPL（从 quant 产物读；有则填，无则 None）
    ppls = read_ppl_from_artifacts(OUT_ROOT, list(available.keys()))

    # 补 FP16 上界 PPL：用 transformers forward 算 fixed-text PPL（与 s5/s6 的 ppl_of 同逻辑）
    if 'FP16' in available and ppls.get('FP16') is None:
        print("补 FP16 PPL（transformers forward，与 s5/s6 同 fixed-text 同逻辑）...")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        fp16_m = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), torch_dtype=torch.float16, device_map="auto")
        fp16_tok = AutoTokenizer.from_pretrained(str(MODEL_DIR))
        fp16_ids = fp16_tok("The future of AI depends on efficient inference at scale.", return_tensors="pt").input_ids.to(fp16_m.device)
        with torch.no_grad():
            fp16_logits = fp16_m(fp16_ids).logits[0]
        fp16_loss = torch.nn.functional.cross_entropy(fp16_logits[:-1], fp16_ids[0,1:])
        ppls['FP16'] = float(torch.exp(fp16_loss).item())
        print(f"  FP16 PPL={ppls['FP16']:.3f}")

    # 维2: downstream gsm8k（每对比点真跑 lm_eval[vllm]）
    table = {}
    for name, path in available.items():
        ds = run_downstream_eval(str(path), tasks=('gsm8k',), num_fewshot=5, limit=250, execute=True)
        gsm8k = None
        if ds['results']:
            try: gsm8k = ds['results']['gsm8k']['exact_match,none']
            except Exception: pass
        table[name] = {'ppl': ppls.get(name), 'gsm8k': gsm8k}
        print(f"  {name:9s} PPL={ppls.get(name)}  gsm8k={gsm8k}")
    print("\n调优收益验证：baseline(掉点) -> tuned/final(追回) -> FP16(上界)")
    # 维3/4: 显存/吞吐（需 vllm serve + /metrics，OUTLINE 3.7 命令）
    print("\n显存/吞吐维：需 vllm serve 后抓 /metrics（kv_cache_usage_perc）+ vllm bench serve")
    print("  curl -s http://localhost:8000/metrics | grep -E 'kv_cache_usage_perc|num_gpu_blocks'")
    json.dump(table, open(OUT_ROOT / 's7_four_dim_table.json', 'w'), indent=2)

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_four_dim_eval()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 命令构造+四向 PPL 聚合逻辑）")

## 产物检查：四向调优收益对比表（finale）

合成 OUTLINE 3.8 四维对比表（质量 PPL+downstream | 显存 | 吞吐 | TTFT）。L3 跑完会在 `out/s7_four_dim_table.json` 产质量维；显存/吞吐维需 vLLM serve 后抓 `/metrics`（命令见 OUTLINE 3.7）。这张表展示 SmoothQuant 工业调优闭环的完整收益。


In [ ]:
import json
p = OUT_ROOT / 's7_four_dim_table.json'
if p.exists():
    table = json.loads(p.read_text())
    print("=== 四向调优收益对比表（baseline→tuned→final→FP16）===")
    print(f"{'对比点':<10}{'PPL':>8}{'gsm8k':>10}")
    # 按 baseline/tuned/final/FP16 顺序展示调优收益趋势
    for name in ['baseline', 'tuned', 'final', 'FP16']:
        if name not in table: continue
        d = table[name]
        ppl = f"{d['ppl']:.2f}" if d.get('ppl') is not None else 'N/A'
        gsm = f"{d['gsm8k']:.3f}" if d.get('gsm8k') is not None else 'N/A'
        print(f"{name:<10}{ppl:>8}{gsm:>10}")
    print("\n显存/吞吐维：见 vllm serve 的 /metrics（kv_cache_usage_perc）+ vllm bench serve")
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
    print("\n[OUTLINE 3.7 抓显存命令示例]")
    print("  curl -s http://localhost:8000/metrics | grep -E 'kv_cache_usage_perc|num_gpu_blocks'")
